# KMeans — Regionais de Saúde de MG por Ano

Unidade de agrupamento: par **(REGIONAL DE SAÚDE, ANO)**. Uma mesma Regional pode cair em agrupamentos diferentes em anos diferentes

O notebook está dividido em duas partes: 
**Parte 1 — Implementação** (entrada, parâmetros, execução do algoritmo e testes de robustez da configuração) 
**Parte 2 — Avaliação e Apresentação** (só resultados: gráficos e caracterização de cada cluster, sem repetir o método).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

sys.path.append(str(Path.cwd()))

from utilitarios_agrupamento_kmeans import (
    COLUNA_ANO, COLUNA_REGIONAL,
    BLOCOS_DE_TAXA, PASTA_TAXA,
    LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA,
    USAR_PCA, RAZAO_MAXIMA_ATRIBUTOS_POR_LINHA, VARIANCIA_EXPLICADA_ALVO,
    NUMERO_MAXIMO_DE_COMPONENTES, SEMENTE_ALEATORIA_FIXA,
    USAR_JANELA_COMUM, ANO_INICIAL_COMUM, ANO_FINAL_COMUM,
    K_MINIMO, K_MAXIMO, SEMENTES_DA_BUSCA, EXECUTAR_TESTE_DE_SENSIBILIDADE_DA_SEMENTE,
    MINIMO_DE_ELEMENTOS_POR_CLUSTER, MAXIMO_DE_ITERACOES,
    NUMERO_DE_VARIAVEIS_DISCRIMINANTES,
    montar_painel,
    descartar_colunas_sem_informacao_e_total, descartar_colunas_com_muitos_nulos, tratar_nulos_sem_imputar,
    padronizar_e_reduzir,
    calcular_inercia_por_k, escolher_k_pelo_cotovelo, busca_em_grade_k,
    maximo_de_iteracoes_usadas,
    executar_kmeans, resumir_agrupamento_kmeans, diagnostico_tamanho_dos_clusters,
    montar_resumo_parametros_kmeans, montar_tabela_pertencimento, montar_visao_agregada_regional_ano,
    montar_resumo_por_cluster,
    selecionar_variaveis_mais_discriminantes,
    calcular_caracterizacao_por_mediana,
    tabela_cluster_por_ano, tabela_clusters_por_regional, resumir_cobertura_por_cluster,
    contar_pares_ausentes_por_regional, contar_municipios_por_regional,
    listar_nulos_por_regional, resumir_variaveis_nulas_por_regional,
    populacao_feminina_media_por_regional, montar_tabela_consolidada_ausencia,
    listar_combinacoes_nulas_de_mortalidade, comparar_populacao_de_risco,
    PASTA_SELECAO_TRANSFORMACAO,
    registrar_log,
)
from graficos_agrupamento_kmeans import (
    grafico_cotovelo_inercia,
    grafico_heatmap_regional_ano,
    grafico_mapa_clusters,
    grafico_heatmap_caracterizacao,
    grafico_cluster_por_ano,
)

# Parte 1 — Implementação

Entrada dos dados, parâmetros de configuração, execução do algoritmo e os testes que validam a configuração escolhida. Nenhum resultado final é discutido aqui -- isso fica todo na Parte 2.

## 1. Leitura dos CSVs de taxa e reformatação 

In [2]:
painel = montar_painel(blocos=BLOCOS_DE_TAXA, pasta_taxa=PASTA_TAXA)
#painel.head()

[LOG] tabela_casos_por_faixa_etária_taxa.csv: 9 categoria(s), 1064 par(es) regional-ano.
[LOG] tabela_casos_por_escolaridade_taxa.csv: 8 categoria(s), 728 par(es) regional-ano.
[LOG] tabela_casos_por_raça_e_cor_taxa.csv: 7 categoria(s), 728 par(es) regional-ano.
[LOG] tabela_mortes_por_faixa_etária_taxa.csv: 9 categoria(s), 1288 par(es) regional-ano.
[LOG] tabela_mamografias_por_faixa_etária_taxa.csv: 9 categoria(s), 364 par(es) regional-ano.
[LOG] Painel montado: 1316 par(es) regional-ano x 42 variável(is), 28 regional(is).


In [3]:
print(
    f"Matriz de entrada: {painel.shape[0]} pares Regional-Ano x {painel.shape[1]} variáveis, "
    f"{painel.index.get_level_values(COLUNA_REGIONAL).nunique()} Regionais de Saúde."
)
pd.set_option("display.max_columns", None)
painel.head(10)

Matriz de entrada: 1316 pares Regional-Ano x 42 variáveis, 28 Regionais de Saúde.


casos-faixa_0-9  casos-faixa_10-19  casos-faixa_20-29  \
REGIONAL DE SAÚDE ANO                                                           
alfenas           1979              NaN                NaN                NaN   
                  1980              NaN                NaN                NaN   
                  1981              NaN                NaN                NaN   
                  1982              NaN                NaN                NaN   
                  1983              NaN                NaN                NaN   
                  1984              NaN                NaN                NaN   
                  1985              NaN                NaN                NaN   
                  1986              NaN                NaN                NaN   
                  1987              NaN                NaN                NaN   
                  1988              NaN                NaN                NaN   

                        casos-faixa_30-39  casos-faixa_40-49  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   
                  1984                NaN                NaN   
                  1985                NaN                NaN   
                  1986                NaN                NaN   
                  1987                NaN                NaN   
                  1988                NaN                NaN   

                        casos-faixa_50-59  casos-faixa_60-69  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   
                  1984                NaN                NaN   
                  1985                NaN                NaN   
                  1986                NaN                NaN   
                  1987                NaN                NaN   
                  1988                NaN                NaN   

                        casos-faixa_70-ou-mais  casos-faixa_total  \
REGIONAL DE SAÚDE ANO                                               
alfenas           1979                     NaN                NaN   
                  1980                     NaN                NaN   
                  1981                     NaN                NaN   
                  1982                     NaN                NaN   
                  1983                     NaN                NaN   
                  1984                     NaN                NaN   
                  1985                     NaN                NaN   
                  1986                     NaN                NaN   
                  1987                     NaN                NaN   
                  1988                     NaN                NaN   

                        casos-escolaridade_fundamental-completo  \
REGIONAL DE SAÚDE ANO                                             
alfenas           1979                                      NaN   
                  1980                                      NaN   
                  1981                                      NaN   
                  1982                                      NaN   
                  1983                                      NaN   
                  1984                                      NaN   
                  1985                                      NaN   
                  1986                                      NaN   
              

In [4]:
matriz_exibicao = descartar_colunas_sem_informacao_e_total(painel).reset_index()
print(
    f"Matriz de entrada (Ano como coluna; sem colunas 'total'/'sem informação'): "
    f"{matriz_exibicao.shape[0]} linhas x {matriz_exibicao.shape[1]} colunas, "
    f"{matriz_exibicao[COLUNA_REGIONAL].nunique()} Regionais de Saúde."
)
matriz_exibicao.head(10)

[LOG] Descartada(s) 7 coluna(s) sem-informação/total: ['casos-faixa_total', 'casos-escolaridade_sem-informacao', 'casos-escolaridade_total', 'casos-raca-cor_sem-informacao', 'casos-raca-cor_total', 'mortalidade_total', 'mamografia_total']
Matriz de entrada (Ano como coluna; sem colunas 'total'/'sem informação'): 1316 linhas x 37 colunas, 28 Regionais de Saúde.


,REGIONAL DE SAÚDE,ANO,casos-faixa_0-9,casos-faixa_10-19,casos-faixa_20-29,casos-faixa_30-39,casos-faixa_40-49,casos-faixa_50-59,casos-faixa_60-69,casos-faixa_70-ou-mais,casos-escolaridade_fundamental-completo,casos-escolaridade_fundamental-incompleto,casos-escolaridade_nenhuma,casos-escolaridade_nivel-medio,casos-escolaridade_nivel-superior-completo,casos-escolaridade_nivel-superior-incompleto,casos-raca-cor_amarela,casos-raca-cor_branca,casos-raca-cor_indigena,casos-raca-cor_parda,casos-raca-cor_preta,mortalidade_0-9,mortalidade_10-19,mortalidade_20-29,mortalidade_30-39,mortalidade_40-49,mortalidade_50-59,mortalidade_60-69,mortalidade_70-ou-mais,mamografia_0-9,mamografia_10-19,mamografia_20-29,mamografia_30-39,mamografia_40-49,mamografia_50-59,mamografia_60-69,mamografia_70-ou-mais
0,alfenas,1979,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.37,28.32,14.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,alfenas,1980,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.96,21.25,55.53,41.03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,alfenas,1981,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.89,6.94,27.26,13.27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,alfenas,1982,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.93,NaN,17.83,NaN,60.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,alfenas,1983,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.82,19.98,8.75,50.06,76.91,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,alfenas,1984,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.05,25.76,36.43,110.58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,alfenas,1985,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.39,42.12,11.78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,alfenas,1986,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.71,18.78,41.32,11.44,33.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,alfenas,1987,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.84,6.13,NaN,33.29,64.86,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,alfenas,1988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.91,43.07,77.70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Janela temporal, descarte de colunas e tratamento de nulos


In [5]:
if USAR_JANELA_COMUM:
    anos = painel.index.get_level_values(COLUNA_ANO)
    matriz = painel[(anos >= ANO_INICIAL_COMUM) & (anos <= ANO_FINAL_COMUM)]
else:
    registrar_log("USAR_JANELA_COMUM=False: usando o período completo de cada bloco, sem restringir ano.")
    matriz = painel

matriz = descartar_colunas_sem_informacao_e_total(matriz)
matriz = descartar_colunas_com_muitos_nulos(matriz, limiar_maximo=LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA)
matriz_antes_do_dropna = matriz
matriz = tratar_nulos_sem_imputar(matriz)

registrar_log(f"Matriz final: {matriz.shape[0]} linha(s) x {matriz.shape[1]} atributo(s).")
matriz.head()

[LOG] Descartada(s) 7 coluna(s) sem-informação/total: ['casos-faixa_total', 'casos-escolaridade_sem-informacao', 'casos-escolaridade_total', 'casos-raca-cor_sem-informacao', 'casos-raca-cor_total', 'mortalidade_total', 'mamografia_total']
[LOG] Descartada(s) 6 coluna(s) com mais de 5% de nulos: ['casos-faixa_0-9', 'casos-faixa_10-19', 'mortalidade_0-9', 'mortalidade_10-19', 'mortalidade_20-29', 'mortalidade_30-39']
[LOG] Descartada(s) 24 de 280 linha(s) (8.6%) por ter ao menos um valor ausente -- nenhum valor foi imputado.
[LOG] Matriz final: 256 linha(s) x 29 atributo(s).


casos-faixa_20-29  casos-faixa_30-39  \
REGIONAL DE SAÚDE ANO                                          
alfenas           2014              11.27              86.41   
                  2015              11.37              76.55   
                  2016              14.35              81.29   
                  2017               5.80              44.51   
                  2018               0.00              49.72   

                        casos-faixa_40-49  casos-faixa_50-59  \
REGIONAL DE SAÚDE ANO                                          
alfenas           2014             116.97             160.58   
                  2015             150.01             204.35   
                  2016             118.78             165.55   
                  2017             109.05             138.38   
                  2018             132.57             118.94   

                        casos-faixa_60-69  casos-faixa_70-ou-mais  \
REGIONAL DE SAÚDE ANO                                               
alfenas           2014             241.86                  261.18   
                  2015             177.21                  251.93   
                  2016             194.96                  283.08   
                  2017             159.53                  214.66   
                  2018             140.00                  147.80   

                        casos-escolaridade_fundamental-completo  \
REGIONAL DE SAÚDE ANO                                             
alfenas           2014                                    20.72   
                  2015                                    23.77   
                  2016                                    19.04   
                  2017                                    26.96   
                  2018                                    34.26   

                        casos-escolaridade_fundamental-incompleto  \
REGIONAL DE SAÚDE ANO                                               
alfenas           2014                                      32.23   
                  2015                                      40.22   
                  2016                                      39.90   
                  2017                                      19.32   
                  2018                                      10.23   

                        casos-escolaridade_nenhuma  \
REGIONAL DE SAÚDE ANO                                
alfenas           2014                        4.14   
                  2015                        4.57   
                  2016                        3.17   
                  2017                        2.70   
                  2018                        1.33   

                        casos-escolaridade_nivel-medio  \
REGIONAL DE SAÚDE ANO                                    
alfenas           2014                           12.43   
                  2015                            9.60   
                  2016                           10.88   
                  2017                           12.58   
                  2018                            9.34   

                        casos-escolaridade_nivel-superior-completo  \
REGIONAL DE SAÚDE ANO                                                
alfenas           2014                                       11.51   
                  2015                                       11.43   
                  2016                                       12.69   
                  2017                                        8.09   
                  2018                                       10.23   

                        casos-escolaridade_nivel-superior-incompleto  \
REGIONAL DE SAÚDE ANO                                                  
alfenas           2014                                          0.46   
                  2015                                          1.37   
                  2016                                          0.45   
                  2017                             

### Diagnóstico: o que foi removido, explicitamente

A etapa acima só loga contagens. Aqui vão as identidades: quais colunas caíram (sem-informação/total + esparsas) e, de cada linha (par Regional-Ano) descartada pelo `dropna`, quais colunas específicas estavam nulas nela -- para conferir que o descarte é coerente com a cobertura real dos CSVs de origem, não um bug de junção.

In [6]:
colunas_descartadas = sorted(set(painel.columns) - set(matriz_antes_do_dropna.columns))
print(f"Colunas descartadas (sem-informação/total + >{LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA:.0%} de nulos): {len(colunas_descartadas)} de {painel.shape[1]}")
display(pd.DataFrame({"COLUNA_DESCARTADA": colunas_descartadas}))

linhas_descartadas_idx = matriz_antes_do_dropna.index.difference(matriz.index)
print(f"\nLinhas (pares Regional-Ano) descartadas pelo dropna: {len(linhas_descartadas_idx)} de {len(matriz_antes_do_dropna)}")

mascara_nulos = matriz_antes_do_dropna.loc[linhas_descartadas_idx].isna()
colunas_nulas_por_linha = mascara_nulos.apply(lambda linha: ", ".join(linha.index[linha]), axis=1)
tabela_linhas_descartadas = colunas_nulas_por_linha.reset_index()
tabela_linhas_descartadas.columns = [COLUNA_REGIONAL, COLUNA_ANO, "COLUNAS_COM_NULO"]
tabela_linhas_descartadas

Colunas descartadas (sem-informação/total + >5% de nulos): 13 de 42


,COLUNA_DESCARTADA
0,casos-escolaridade_sem-informacao
1,casos-escolaridade_total
2,casos-faixa_0-9
3,casos-faixa_10-19
4,casos-faixa_total
5,casos-raca-cor_sem-informacao
6,casos-raca-cor_total
7,mamografia_total
8,mortalidade_0-9
9,mortalidade_10-19



Linhas (pares Regional-Ano) descartadas pelo dropna: 24 de 280


,REGIONAL DE SAÚDE,ANO,COLUNAS_COM_NULO
0,diamantina,2017,mortalidade_40-49
1,diamantina,2020,mortalidade_50-59
2,diamantina,2022,mortalidade_40-49
3,ituiutaba,2017,mortalidade_40-49
4,ituiutaba,2018,mortalidade_50-59
5,ituiutaba,2022,mortalidade_40-49
6,ituiutaba,2023,mortalidade_40-49
7,pedra azul,2014,mortalidade_70-ou-mais
8,pedra azul,2015,mortalidade_40-49
9,pedra azul,2018,mortalidade_50-59


### Diagnóstico: proporção de nulos por coluna e cobertura da mamografia em 2013

Proporção de nulos de cada coluna na janela comum, **antes** do descarte por limiar (já sem as colunas de total e sem informação), e a cobertura da base de mamografia em 2013 -- que justifica iniciar a janela em 2014.

In [7]:
anos_painel = painel.index.get_level_values(COLUNA_ANO)
matriz_janela = descartar_colunas_sem_informacao_e_total(
    painel[(anos_painel >= ANO_INICIAL_COMUM) & (anos_painel <= ANO_FINAL_COMUM)]
)
proporcao_de_nulos = matriz_janela.isna().mean().sort_values(ascending=False)
print(f"Proporção de nulos por coluna em {ANO_INICIAL_COMUM}-{ANO_FINAL_COMUM}, antes do descarte (só colunas com algum nulo):")
display(proporcao_de_nulos[proporcao_de_nulos > 0].rename("PROPORCAO_DE_NULOS").to_frame())

colunas_mamografia = []
for coluna in painel.columns:
    if coluna.startswith("mamografia_") and not coluna.endswith("_total"):
        colunas_mamografia.append(coluna)
mamografia_2013 = painel[anos_painel == 2013][colunas_mamografia]
print(
    f"Mamografia em 2013: {mamografia_2013.notna().any(axis=1).sum()} de {len(mamografia_2013)} "
    "Regionais com algum registro. Proporção de nulos por faixa:"
)
mamografia_2013.isna().mean().rename("PROPORCAO_DE_NULOS_2013").to_frame()

[LOG] Descartada(s) 7 coluna(s) sem-informação/total: ['casos-faixa_total', 'casos-escolaridade_sem-informacao', 'casos-escolaridade_total', 'casos-raca-cor_sem-informacao', 'casos-raca-cor_total', 'mortalidade_total', 'mamografia_total']
Proporção de nulos por coluna em 2014-2023, antes do descarte (só colunas com algum nulo):


,PROPORCAO_DE_NULOS
mortalidade_0-9,1.000000
mortalidade_10-19,0.992857
mortalidade_20-29,0.767857
casos-faixa_0-9,0.400000
mortalidade_30-39,0.132143
casos-faixa_10-19,0.100000
mortalidade_40-49,0.042857
mortalidade_50-59,0.021429
mortalidade_70-ou-mais,0.010714
mortalidade_60-69,0.007143


Mamografia em 2013: 3 de 28 Regionais com algum registro. Proporção de nulos por faixa:


,PROPORCAO_DE_NULOS_2013
mamografia_0-9,1.000000
mamografia_10-19,1.000000
mamografia_20-29,1.000000
mamografia_30-39,0.892857
mamografia_40-49,0.892857
mamografia_50-59,0.892857
mamografia_60-69,0.892857
mamografia_70-ou-mais,0.892857


## Investigação complementar: por que Pirapora e Ituiutaba têm poucos pares (Regional, Ano)

Pirapora e Ituiutaba aparecem com poucos pares (Regional, Ano) no conjunto final de 256 observações. Esta seção investiga a causa técnica, em 6 passos, sem alterar o pipeline de limpeza (o `dropna` de `tratar_nulos_sem_imputar`, acima, permanece como está).

### 1. Quantificação da ausência, por Regional

In [8]:
tabela_ausencia_por_regional = contar_pares_ausentes_por_regional(matriz_antes_do_dropna, matriz)
tabela_ausencia_por_regional

,REGIONAL DE SAÚDE,ANOS_ESPERADOS,PARES_NO_FINAL,PARES_AUSENTES,PROPORCAO_AUSENTE
0,ituiutaba,10,6,4,0.4
1,pedra azul,10,6,4,0.4
2,pirapora,10,6,4,0.4
3,unai,10,6,4,0.4
4,diamantina,10,7,3,0.3
5,teofilo otoni,10,8,2,0.2
6,ponte nova,10,9,1,0.1
7,sao joao del rei,10,9,1,0.1
8,sete lagoas,10,9,1,0.1
9,alfenas,10,10,0,0.0


Quatro Regionais empatam no topo da ausência -- Pirapora, Ituiutaba, Pedra Azul e Unaí, cada uma com 4 de 10 pares (Regional, Ano) ausentes (40%). Pirapora e Ituiutaba estão, portanto, entre as Regionais mais afetadas, mas não isoladas -- o padrão se repete em outras Regionais pequenas (ver passo 2).

### 2. Atributo estrutural: nº de municípios por Regional

Mesma tabela (`tabela_municipios_regionais.csv`) usada no enriquecimento geográfico da EDA.

In [9]:
tabela_municipios_por_regional = contar_municipios_por_regional()
print(f"{len(tabela_municipios_por_regional)} Regionais, ordenadas da com menos para a com mais municípios:")
tabela_municipios_por_regional

28 Regionais, ordenadas da com menos para a com mais municípios:


,REGIONAL DE SAÚDE,N_MUNICIPIOS
0,pirapora,7
1,ituiutaba,9
2,unai,12
3,leopoldina,15
4,uberlandia,18
5,sao joao del rei,19
6,patos de minas,21
7,itabira,24
8,passos,24
9,pedra azul,25


### 3. Origem do nulo, variável por variável (antes do dropna)

Para Pirapora e Ituiutaba, quais das colunas (já agregadas por Regional-Ano, depois do descarte de colunas esparsas e antes do `dropna` de linha) estão nulas em cada ano.

In [10]:
REGIONAIS_INVESTIGADAS = ["pirapora", "ituiutaba"]
tabela_nulos_pirapora_ituiutaba = listar_nulos_por_regional(matriz_antes_do_dropna, REGIONAIS_INVESTIGADAS)
tabela_nulos_pirapora_ituiutaba

,REGIONAL DE SAÚDE,ANO,VARIAVEL
0,ituiutaba,2017,mortalidade_40-49
1,ituiutaba,2018,mortalidade_50-59
2,ituiutaba,2022,mortalidade_40-49
3,ituiutaba,2023,mortalidade_40-49
4,pirapora,2015,mortalidade_70-ou-mais
5,pirapora,2018,mortalidade_40-49
6,pirapora,2020,mortalidade_40-49
7,pirapora,2023,mortalidade_50-59


In [11]:
resumo_variaveis_nulas = resumir_variaveis_nulas_por_regional(tabela_nulos_pirapora_ituiutaba)
resumo_variaveis_nulas

,REGIONAL DE SAÚDE,VARIAVEL,N_OCORRENCIAS
0,ituiutaba,mortalidade_40-49,3
1,ituiutaba,mortalidade_50-59,1
2,pirapora,mortalidade_40-49,2
3,pirapora,mortalidade_50-59,1
4,pirapora,mortalidade_70-ou-mais,1


As 8 ocorrências de nulo de Pirapora e Ituiutaba (4 em cada Regional), na janela 2014-2023, estão **todas** em colunas `mortalidade_*` -- faixas etárias específicas (40-49, 50-59, 70 OU MAIS), nunca em `casos-*` ou `mamografia_*`. É o mesmo padrão documentado na seção 4.4 de `DECISOES.md` (24 de 280 linhas descartadas no total, quase todas por `mortalidade_*`): contagens raras de óbito por faixa etária, não falha de coleta.

### 4. Contagem zero vs. ausência: voltando à fonte bruta

Para **todas as 8** combinações nulas do passo 3 (não apenas uma amostra), volta-se ao CSV bruto (nível município, antes de qualquer agregação por Regional) para checar se a célula está de fato ausente na exportação do Mortalidade Web/INCA (nenhuma linha com valor, nem zero) ou se existe outro padrão.

In [12]:
caminho_mortalidade_bruta = PASTA_SELECAO_TRANSFORMACAO / "Mortalidade" / "tabela_mortes_por_faixa_etária.csv"
mortalidade_bruta = pd.read_csv(caminho_mortalidade_bruta)

combinacoes_nulas_de_mortalidade = listar_combinacoes_nulas_de_mortalidade(tabela_nulos_pirapora_ituiutaba)
print(f"{len(combinacoes_nulas_de_mortalidade)} combinações nulas de mortalidade a checar na fonte bruta:")
display(combinacoes_nulas_de_mortalidade)

resultado_checagem_fonte_bruta = []
for _, linha_combinacao in combinacoes_nulas_de_mortalidade.iterrows():
    regional_bruta = linha_combinacao[COLUNA_REGIONAL]
    ano = linha_combinacao[COLUNA_ANO]
    faixa = linha_combinacao["FAIXA"]
    coluna = f"{ano}_{faixa}"
    sub = mortalidade_bruta.loc[mortalidade_bruta[COLUNA_REGIONAL] == regional_bruta, ["MUNICÍPIO", coluna]]
    print(f"--- {regional_bruta} | {ano} | faixa {faixa} ({len(sub)} município(s) da Regional) ---")
    display(sub)
    todos_nulos = bool(sub[coluna].isna().all())
    print(f"Todos os municípios nulos na célula? {todos_nulos}\n")
    resultado_checagem_fonte_bruta.append({
        COLUNA_REGIONAL: regional_bruta, COLUNA_ANO: ano, "FAIXA": faixa,
        "N_MUNICIPIOS_DA_REGIONAL": len(sub), "TODOS_NULOS_NA_FONTE_BRUTA": todos_nulos,
    })


8 combinações nulas de mortalidade a checar na fonte bruta:


,REGIONAL DE SAÚDE,ANO,FAIXA
0,Ituiutaba,2017,40-49
1,Ituiutaba,2018,50-59
2,Ituiutaba,2022,40-49
3,Ituiutaba,2023,40-49
4,Pirapora,2015,70 OU MAIS
5,Pirapora,2018,40-49
6,Pirapora,2020,40-49
7,Pirapora,2023,50-59


--- Ituiutaba | 2017 | faixa 40-49 (9 município(s) da Regional) ---


,MUNICÍPIO,2017_40-49
295,cachoeira dourada,NaN
296,campina verde,NaN
297,canapolis,NaN
298,capinopolis,NaN
299,centralina,NaN
300,gurinhata,NaN
301,ipiacu,NaN
302,ituiutaba,NaN
303,santa vitoria,NaN


Todos os municípios nulos na célula? True

--- Ituiutaba | 2018 | faixa 50-59 (9 município(s) da Regional) ---


,MUNICÍPIO,2018_50-59
295,cachoeira dourada,NaN
296,campina verde,NaN
297,canapolis,NaN
298,capinopolis,NaN
299,centralina,NaN
300,gurinhata,NaN
301,ipiacu,NaN
302,ituiutaba,NaN
303,santa vitoria,NaN


... (9 outputs left out)

,MUNICÍPIO,2020_40-49
539,buritizeiro,NaN
540,ibiai,NaN
541,lassance,NaN
542,pirapora,NaN
543,ponto chique,NaN
544,santa fe de minas,NaN
545,varzea da palma,NaN


Todos os municípios nulos na célula? True

--- Pirapora | 2023 | faixa 50-59 (7 município(s) da Regional) ---


,MUNICÍPIO,2023_50-59
539,buritizeiro,NaN
540,ibiai,NaN
541,lassance,NaN
542,pirapora,NaN
543,ponto chique,NaN
544,santa fe de minas,NaN
545,varzea da palma,NaN


Todos os municípios nulos na célula? True



In [13]:
tabela_checagem_fonte_bruta = pd.DataFrame(resultado_checagem_fonte_bruta)
print(
    f"Em {tabela_checagem_fonte_bruta['TODOS_NULOS_NA_FONTE_BRUTA'].sum()} de "
    f"{len(tabela_checagem_fonte_bruta)} combinações, 100% dos municípios da Regional "
    "estão nulos na fonte bruta (nenhuma linha com valor, nem zero)."
)
tabela_checagem_fonte_bruta

Em 8 de 8 combinações, 100% dos municípios da Regional estão nulos na fonte bruta (nenhuma linha com valor, nem zero).


,REGIONAL DE SAÚDE,ANO,FAIXA,N_MUNICIPIOS_DA_REGIONAL,TODOS_NULOS_NA_FONTE_BRUTA
0,Ituiutaba,2017,40-49,9,True
1,Ituiutaba,2018,50-59,9,True
2,Ituiutaba,2022,40-49,9,True
3,Ituiutaba,2023,40-49,9,True
4,Pirapora,2015,70 OU MAIS,7,True
5,Pirapora,2018,40-49,7,True
6,Pirapora,2020,40-49,7,True
7,Pirapora,2023,50-59,7,True


Nas 8 combinações, **todos** os municípios da Regional aparecem como `NaN` na célula (ano, faixa), nunca uma mistura de `NaN` com algum município em 0 -- ou seja, a exportação do Mortalidade Web simplesmente não emite uma linha quando a contagem é zero para aquele recorte (município, ano, faixa etária), em vez de emitir o valor `0`. Como a Mortalidade (SIM) é um registro administrativo de cobertura quase universal -- não um levantamento amostral sujeito a não resposta --, a leitura mais provável é contagem genuinamente zero (nenhum óbito por câncer de mama naquela faixa etária estreita, naquele ano, naquela Regional pequena), não um buraco de coleta.

### 5. Comparação com a população de risco

Para as mesmas 8 combinações (Regional, Ano, Faixa) nulas: população feminina da Regional (mesma base interpolada usada como denominador das taxas), mediana estadual na mesma faixa/ano, e a posição da Regional no ranking das 28 (1º = menor população) -- para situar o tamanho relativo, não só o valor absoluto. Por fim, a população feminina total média de cada Regional na janela comum.

In [14]:
tabela_populacao_de_risco = comparar_populacao_de_risco(combinacoes_nulas_de_mortalidade)
tabela_populacao_de_risco

,REGIONAL DE SAÚDE,ANO,FAIXA,POPULACAO_FEMININA_DA_REGIONAL,MEDIANA_ESTADUAL_NA_FAIXA_ANO,POSICAO_NO_RANKING_ESTADUAL
0,Ituiutaba,2017,40-49,13875.0,32296.0,2º de 28 (menor)
1,Ituiutaba,2018,50-59,12465.0,28122.5,2º de 28 (menor)
2,Ituiutaba,2022,40-49,14131.0,33694.5,2º de 28 (menor)
3,Ituiutaba,2023,40-49,14184.0,34029.5,2º de 28 (menor)
4,Pirapora,2015,70 OU MAIS,2971.0,14060.5,1º de 28 (menor)
5,Pirapora,2018,40-49,9414.0,32546.5,1º de 28 (menor)
6,Pirapora,2020,40-49,9655.0,33055.5,1º de 28 (menor)
7,Pirapora,2023,50-59,8228.0,31020.0,1º de 28 (menor)


In [15]:
tabela_populacao_feminina_media_todas_regionais = populacao_feminina_media_por_regional(
    tabela_municipios_por_regional[COLUNA_REGIONAL].tolist(), ANO_INICIAL_COMUM, ANO_FINAL_COMUM,
)
tabela_populacao_feminina_media = tabela_populacao_feminina_media_todas_regionais[
    tabela_populacao_feminina_media_todas_regionais[COLUNA_REGIONAL].isin(REGIONAIS_INVESTIGADAS)
].reset_index(drop=True)
tabela_populacao_feminina_media

,REGIONAL DE SAÚDE,POPULACAO_FEMININA_MEDIA
0,pirapora,65066.9
1,ituiutaba,93714.4


### 6. Tabela consolidada

Regional | nº de municípios (+ ranking entre as 28) | população feminina média (+ ranking entre as 28) | % de pares (Regional, Ano) ausentes | principais variáveis nulas.

In [16]:
tabela_consolidada_ausencia = montar_tabela_consolidada_ausencia(
    tabela_ausencia_por_regional, tabela_municipios_por_regional,
    tabela_populacao_feminina_media_todas_regionais,
    tabela_nulos_pirapora_ituiutaba, REGIONAIS_INVESTIGADAS,
)
tabela_consolidada_ausencia

,REGIONAL DE SAÚDE,N_MUNICIPIOS,RANKING_N_MUNICIPIOS,POPULACAO_FEMININA_MEDIA,RANKING_POPULACAO_FEMININA,PROPORCAO_AUSENTE,PRINCIPAIS_VARIAVEIS_NULAS
0,ituiutaba,9,2º de 28 (menor),93714.4,2º de 28 (menor),0.4,"mortalidade_40-49 (3x), mortalidade_50-59 (1x)"
1,pirapora,7,1º de 28 (menor),65066.9,1º de 28 (menor),0.4,"mortalidade_40-49 (2x), mortalidade_50-59 (1x)..."


## 3. Padronização (RobustScaler)

Cada variável é centrada na mediana e dividida pela amplitude interquartil (IQR). O KMeans roda diretamente sobre as 29 variáveis padronizadas.

In [17]:
matriz_padronizada, matriz_para_kmeans, pca, variancia_acumulada = padronizar_e_reduzir(
    matriz,
    razao_maxima_atributos_por_linha=RAZAO_MAXIMA_ATRIBUTOS_POR_LINHA,
    variancia_alvo=VARIANCIA_EXPLICADA_ALVO,
    numero_maximo_de_componentes=NUMERO_MAXIMO_DE_COMPONENTES,
    semente=SEMENTE_ALEATORIA_FIXA,
    usar_pca=USAR_PCA,
)

[LOG] PCA não aplicado: USAR_PCA=False (forçado; razão atributos/linhas = 0.113).


## 4. Escolha de k (cotovelo da curva de inércia)

In [18]:
tabela_k = calcular_inercia_por_k(matriz_para_kmeans, k_minimo=K_MINIMO, k_maximo=K_MAXIMO, semente=SEMENTE_ALEATORIA_FIXA)
k_escolhido, indice_do_cotovelo = escolher_k_pelo_cotovelo(tabela_k)

figura_cotovelo = grafico_cotovelo_inercia(tabela_k, indice_do_cotovelo, k_escolhido)
figura_cotovelo.show()
tabela_k

[LOG] Inércia calculada para k = 2..12 (11 valor(es) de k).
[LOG] Cotovelo da curva de inércia na posição 3 de 11 -- k = 5


,k,inercia,n_iter_
0,2,5151.241069,8
1,3,4492.131216,11
2,4,4060.881057,14
3,5,3743.330815,12
4,6,3560.265650,7
5,7,3417.892652,13
6,8,3266.809198,19
7,9,3140.871574,15
8,10,3051.904629,13
9,11,2946.795863,21


### Busca em grade

In [19]:
# Teste de sensibilidade à semente: não utilizado no TCC (o pipeline principal usa só SEMENTE_ALEATORIA_FIXA)
if EXECUTAR_TESTE_DE_SENSIBILIDADE_DA_SEMENTE:
    tabela_da_busca = busca_em_grade_k(matriz_para_kmeans, k_minimo=K_MINIMO, k_maximo=K_MAXIMO, sementes=SEMENTES_DA_BUSCA)
    print(f"Resultados da busca em grade: k = {K_MINIMO}..{min(K_MAXIMO, len(matriz_para_kmeans) - 1)}, sementes = {SEMENTES_DA_BUSCA} ({len(tabela_da_busca)} combinações).")

    resumo_por_k = tabela_da_busca.groupby("k").agg(
        inercia_media=("inercia", "mean"), inercia_desvio=("inercia", "std"),
    ).reset_index()
    print("\nEstabilidade por k através das sementes testadas:")
    display(resumo_por_k)
else:
    tabela_da_busca = None
    print("Teste de sensibilidade à semente: não utilizado no TCC (EXECUTAR_TESTE_DE_SENSIBILIDADE_DA_SEMENTE=False).")

Teste de sensibilidade à semente: não utilizado no TCC (EXECUTAR_TESTE_DE_SENSIBILIDADE_DA_SEMENTE=False).


In [20]:
if EXECUTAR_TESTE_DE_SENSIBILIDADE_DA_SEMENTE:
    print(f"Variação entre sementes para o k escolhido (k = {k_escolhido}):")
    display(tabela_da_busca[tabela_da_busca["k"] == k_escolhido][["inercia"]].agg(["min", "max"]))
else:
    print("Teste de sensibilidade à semente: não utilizado no TCC (EXECUTAR_TESTE_DE_SENSIBILIDADE_DA_SEMENTE=False).")

Teste de sensibilidade à semente: não utilizado no TCC (EXECUTAR_TESTE_DE_SENSIBILIDADE_DA_SEMENTE=False).


In [21]:
if EXECUTAR_TESTE_DE_SENSIBILIDADE_DA_SEMENTE:
    inercia_por_semente = tabela_da_busca.loc[tabela_da_busca["k"] == k_escolhido, "inercia"]
    desvio_inercia = inercia_por_semente.std()
    media_inercia = inercia_por_semente.mean()
    cv_inercia = desvio_inercia / media_inercia * 100

    print(f"Coeficiente de variação da inércia entre as {len(inercia_por_semente)} sementes (k = {k_escolhido}):")
    print(f"  S  = {desvio_inercia:.4f}")
    print(f"  x̄  = {media_inercia:.4f}")
    print(f"  CV = {cv_inercia:.4f}%")
else:
    print("Teste de sensibilidade à semente: não utilizado no TCC (EXECUTAR_TESTE_DE_SENSIBILIDADE_DA_SEMENTE=False).")

Teste de sensibilidade à semente: não utilizado no TCC (EXECUTAR_TESTE_DE_SENSIBILIDADE_DA_SEMENTE=False).


In [22]:
# Ponto onde a seleção dependia de agregar várias sementes (tabela_da_busca, teste de sensibilidade desativado
# acima) -- agora usa só tabela_k, calculada com a semente fixa (SEMENTE_ALEATORIA_FIXA).
maximo_iteracoes = maximo_de_iteracoes_usadas(tabela_k)
max_iter_configurado = MAXIMO_DE_ITERACOES
print(
    f"Máximo de iterações até convergência entre todas as execuções de KMeans "
    f"(cotovelo, semente fixa): {maximo_iteracoes} (max_iter configurado: {max_iter_configurado})."
)


Máximo de iterações até convergência entre todas as execuções de KMeans (cotovelo, semente fixa): 21 (max_iter configurado: 30).


## 5. KMeans

In [23]:
rotulos, modelo_kmeans = executar_kmeans(matriz_para_kmeans, k_escolhido, semente=SEMENTE_ALEATORIA_FIXA)
n_clusters, inercia_final = resumir_agrupamento_kmeans(matriz_para_kmeans, rotulos, modelo_kmeans)

registrar_log(f"KMeans(k={k_escolhido}, random_state={SEMENTE_ALEATORIA_FIXA})")
registrar_log(f"  Agrupamentos: {n_clusters}")
registrar_log(f"  Inércia (WCSS): {inercia_final:.4f}")

_ = diagnostico_tamanho_dos_clusters(rotulos, minimo=MINIMO_DE_ELEMENTOS_POR_CLUSTER)

[LOG] KMeans(k=5, random_state=42)
[LOG]   Agrupamentos: 5
[LOG]   Inércia (WCSS): 3743.3308


# Parte 2 — Avaliação e Apresentação (Resultados)


## Parâmetros finais e atributos usados

In [24]:
resumo_parametros = montar_resumo_parametros_kmeans(
    k_escolhido, inercia_final,
    usar_robust_scaler=True, pca=pca, variancia_acumulada=variancia_acumulada,
    n_linhas=matriz.shape[0], n_atributos=matriz.shape[1], n_clusters=n_clusters,
)
resumo_parametros.drop(columns=["PCA_APLICADO", "PCA_N_COMPONENTES", "PCA_VARIANCIA_EXPLICADA"])

,K,ESCALONADOR,N_LINHAS,N_ATRIBUTOS,N_CLUSTERS,INERCIA
0,5,RobustScaler,256,29,5,3743.330815


In [25]:
atributos_usados = pd.DataFrame({"ATRIBUTO": list(matriz.columns)})
atributos_usados

,ATRIBUTO
0,casos-faixa_20-29
1,casos-faixa_30-39
2,casos-faixa_40-49
3,casos-faixa_50-59
4,casos-faixa_60-69
5,casos-faixa_70-ou-mais
6,casos-escolaridade_fundamental-completo
7,casos-escolaridade_fundamental-incompleto
8,casos-escolaridade_nenhuma
9,casos-escolaridade_nivel-medio


## Visualizações

In [26]:
resultado = montar_tabela_pertencimento(matriz.index, rotulos)

### Composição de cada cluster (tabela, não só o gráfico)


In [27]:
resumo_por_cluster = montar_resumo_por_cluster(resultado)
resumo_por_cluster

,CLUSTER,N_PARES,N_REGIONAIS_DISTINTAS,MEMBROS
0,1,45,16,"alfenas (2017), alfenas (2019), barbacena (201..."
1,2,61,13,"diamantina (2014), diamantina (2015), diamanti..."
2,3,97,19,"barbacena (2014), barbacena (2015), barbacena ..."
3,4,49,12,"alfenas (2014), alfenas (2015), alfenas (2016)..."
4,5,4,1,"passos (2019), passos (2021), passos (2022), p..."


## Mapa: cluster por Regional e Ano

In [28]:
resultado

,REGIONAL DE SAÚDE,ANO,CLUSTER
0,alfenas,2014,4
1,alfenas,2015,4
2,alfenas,2016,4
3,alfenas,2017,1
4,alfenas,2018,4
...,...,...,...
251,varginha,2019,3
252,varginha,2020,3
253,varginha,2021,3
254,varginha,2022,3


In [29]:
visao_agregada_regional_ano = montar_visao_agregada_regional_ano(resultado)
visao_agregada_regional_ano

ANO,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
REGIONAL DE SAÚDE,,,,,,,,,,
alfenas,4.0,4.0,4.0,1.0,4.0,1.0,4.0,4.0,4.0,4.0
barbacena,3.0,3.0,1.0,4.0,1.0,1.0,3.0,3.0,1.0,4.0
belo horizonte,3.0,3.0,3.0,1.0,1.0,1.0,3.0,3.0,3.0,3.0
coronel fabriciano,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
diamantina,2.0,2.0,2.0,NaN,2.0,2.0,NaN,2.0,NaN,2.0
divinopolis,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,4.0,3.0
governador valadares,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
itabira,3.0,2.0,1.0,1.0,2.0,2.0,3.0,3.0,4.0,4.0
ituiutaba,2.0,3.0,3.0,NaN,NaN,3.0,2.0,2.0,NaN,NaN


### Heatmap: cluster por Regional e Ano

In [30]:
figura_heatmap = grafico_heatmap_regional_ano(visao_agregada_regional_ano)
figura_heatmap.show()

### Mapa espacial dos clusters — recorte de 2023

Reaproveita a mesma malha geográfica das Regionais de Saúde usada na EDA (`geobr.read_municipality` + `geopandas.dissolve`, ver `Avaliação e Apresentação/mapa_regional_valores_absolutos.ipynb`). Mostra o cluster de cada Regional em **2023** (último ano com dado) — um recorte de um único ano, não a série completa. A evolução de cada Regional ao longo de todos os anos está no heatmap cluster × (Regional, Ano) acima.

In [31]:
import json as _json

import geobr

from utilitarios_comuns import COLUNA_CODIGO_IBGE, PASTA_SELECAO_TRANSFORMACAO, normalizar_codigo_ibge

ANO_MALHA_MUNICIPIOS_MG = 2024
TOLERANCIA_SOLDA_TOPOLOGIA_METROS = 100
TOLERANCIA_SIMPLIFICACAO_GEOMETRIA_METROS = 500
CAMINHO_MUNICIPIOS_REGIONAIS = PASTA_SELECAO_TRANSFORMACAO / "Municípios" / "tabela_municipios_regionais.csv"

mg_municipios = geobr.read_municipality(code_muni="MG", year=ANO_MALHA_MUNICIPIOS_MG).to_crs(4326)
mg_municipios[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(mg_municipios["code_muni"])

tabela_municipios_regionais = pd.read_csv(CAMINHO_MUNICIPIOS_REGIONAIS)
tabela_municipios_regionais[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(
    tabela_municipios_regionais[COLUNA_CODIGO_IBGE]
)
tabela_municipios_regionais = tabela_municipios_regionais.rename(
    columns={"REGIONAL NORMALIZADA": COLUNA_REGIONAL}
)

mg_com_regional = mg_municipios.merge(
    tabela_municipios_regionais[[COLUNA_CODIGO_IBGE, COLUNA_REGIONAL]],
    on=COLUNA_CODIGO_IBGE, how="left",
)

municipios_sem_regional = mg_com_regional[COLUNA_REGIONAL].isna()
mg_projetado = mg_com_regional.to_crs(epsg=31983)
if municipios_sem_regional.any():
    registrar_log(
        f"{int(municipios_sem_regional.sum())} município(s) sem Regional mapeada -- "
        f"preenchendo pela Regional do vizinho mais próximo (só para fins de geometria)."
    )
    com_regional = mg_projetado[~municipios_sem_regional]
    for indice in mg_projetado[municipios_sem_regional].index:
        geometria = mg_projetado.loc[indice, "geometry"]
        distancias = com_regional.geometry.distance(geometria)
        mg_projetado.loc[indice, COLUNA_REGIONAL] = com_regional.loc[distancias.idxmin(), COLUNA_REGIONAL]

# Buffer antes do dissolve para fechar falhas de topologia entre municípios; é desfeito depois
mg_projetado["geometry"] = mg_projetado.buffer(TOLERANCIA_SOLDA_TOPOLOGIA_METROS)

geometria_regionais_projetada = mg_projetado.dissolve(by=COLUNA_REGIONAL, as_index=False)[[COLUNA_REGIONAL, "geometry"]]
geometria_regionais_projetada["geometry"] = geometria_regionais_projetada.buffer(-TOLERANCIA_SOLDA_TOPOLOGIA_METROS)

geometria_regionais = geometria_regionais_projetada.to_crs(4326)

if TOLERANCIA_SIMPLIFICACAO_GEOMETRIA_METROS > 0:
    geometria_simplificada = geometria_regionais_projetada.copy()
    geometria_simplificada["geometry"] = geometria_simplificada.simplify(
        TOLERANCIA_SIMPLIFICACAO_GEOMETRIA_METROS, preserve_topology=True
    )
    geometria_simplificada = geometria_simplificada.to_crs(4326)
else:
    geometria_simplificada = geometria_regionais_projetada.to_crs(4326)

geojson_regionais = _json.loads(geometria_simplificada.to_json())
registrar_log(f"Geometria dissolvida: {len(geometria_regionais)} regionais de saúde.")

[LOG] Geometria dissolvida: 28 regionais de saúde.


In [32]:
ano_mapa_clusters = resultado[COLUNA_ANO].max()
tabela_cluster_ano_referencia = resultado[resultado[COLUNA_ANO] == ano_mapa_clusters]

figura_mapa_clusters = grafico_mapa_clusters(
    geojson_regionais, geometria_regionais, tabela_cluster_ano_referencia, ano_referencia=int(ano_mapa_clusters),
)
figura_mapa_clusters.show()

## Caracterização dos clusters por mediana

Para cada variável (valores originais, antes do `RobustScaler`) e cada cluster, compara-se a mediana do cluster com a mediana geral, normalizada pelo IQR geral:

```
d_cj = (Med_cj − Med_j) / IQR_j
```

`Med_cj` = mediana da variável j no cluster c; `Med_j` = mediana geral da variável j (256 pares); `IQR_j` = IQR geral da variável j (Q75 − Q25). É descritivo, sem teste de significância -- a mediana (em vez da média) é mais robusta a outliers (ex.: Belo Horizonte, que puxa a média mas não a mediana). As variáveis de cada cluster são ordenadas por `|d_cj|`; o sinal indica se o cluster está acima (`d_cj > 0`) ou abaixo (`d_cj < 0`) da mediana geral. Quando o IQR geral de uma variável é 0 (quartis 1, 2 e 3 coincidentes), `d_cj` fica indefinido (`0/0`) e a variável é excluída do ranking.

In [33]:
caracterizacao_por_mediana = calcular_caracterizacao_por_mediana(matriz, rotulos)
variaveis_de_maior_d_cj = selecionar_variaveis_mais_discriminantes(
    caracterizacao_por_mediana, top_n=NUMERO_DE_VARIAVEIS_DISCRIMINANTES
)

for cluster in sorted(variaveis_de_maior_d_cj["CLUSTER"].unique()):
    print(f"\n=== Cluster {cluster} — maiores |d_cj| ===")
    display(
        variaveis_de_maior_d_cj[variaveis_de_maior_d_cj["CLUSTER"] == cluster]
        .drop(columns="CLUSTER")
        .reset_index(drop=True)
    )

[LOG] ATENÇÃO: 1 variável(is) com IQR geral = 0 (sem d_cj): ['casos-raca-cor_indigena']

=== Cluster 1 — maiores |d_cj| ===


,VARIAVEL,MEDIANA_CLUSTER,MEDIANA_GERAL,IQR_GERAL,D_CJ
0,mamografia_30-39,1045.06,466.970,413.0150,1.399683
1,mamografia_70-ou-mais,3569.41,1620.410,1594.4175,1.222390
2,mamografia_40-49,7933.67,3865.745,3394.8000,1.198281
3,mamografia_20-29,85.02,40.475,51.2575,0.869044
4,mamografia_60-69,15956.18,10264.895,6917.0950,0.822785
5,mamografia_50-59,18155.09,12338.030,8389.2475,0.693395
6,mamografia_10-19,9.96,4.455,7.9425,0.693107
7,mamografia_0-9,7.40,3.050,7.6850,0.566038
8,casos-faixa_20-29,2.99,4.710,5.3850,-0.319406
9,mortalidade_70-ou-mais,72.81,63.020,34.2125,0.286153



=== Cluster 2 — maiores |d_cj| ===


,VARIAVEL,MEDIANA_CLUSTER,MEDIANA_GERAL,IQR_GERAL,D_CJ
0,casos-faixa_60-69,85.08,161.775,69.2300,-1.107829
1,casos-faixa_70-ou-mais,81.37,159.985,94.0775,-0.835641
2,casos-faixa_50-59,90.09,133.600,57.9800,-0.750431
3,mortalidade_70-ou-mais,37.44,63.020,34.2125,-0.747680
4,casos-faixa_40-49,62.04,93.765,43.2125,-0.734163
5,casos-raca-cor_branca,7.59,26.830,31.1550,-0.617557
6,casos-faixa_30-39,23.93,39.870,25.8450,-0.616754
7,casos-escolaridade_fundamental-incompleto,11.57,20.215,14.4475,-0.598373
8,casos-escolaridade_nivel-superior-completo,3.85,7.595,6.3275,-0.591861
9,casos-escolaridade_fundamental-completo,4.07,7.990,7.0025,-0.559800



=== Cluster 3 — maiores |d_cj| ===


,VARIAVEL,MEDIANA_CLUSTER,MEDIANA_GERAL,IQR_GERAL,D_CJ
0,casos-raca-cor_amarela,0.26,0.000,0.5000,0.520000
1,casos-raca-cor_parda,29.49,21.895,20.7100,0.366731
2,mortalidade_60-69,41.46,35.110,20.3675,0.311771
3,mamografia_60-69,8278.82,10264.895,6917.0950,-0.287126
4,mamografia_50-59,10104.90,12338.030,8389.2475,-0.266190
5,mamografia_20-29,28.33,40.475,51.2575,-0.236941
6,mamografia_40-49,3124.26,3865.745,3394.8000,-0.218418
7,mamografia_30-39,380.75,466.970,413.0150,-0.208758
8,mortalidade_70-ou-mais,69.86,63.020,34.2125,0.199927
9,mamografia_70-ou-mais,1318.10,1620.410,1594.4175,-0.189605



=== Cluster 4 — maiores |d_cj| ===


,VARIAVEL,MEDIANA_CLUSTER,MEDIANA_GERAL,IQR_GERAL,D_CJ
0,casos-escolaridade_nivel-medio,21.31,10.895,8.3875,1.241729
1,casos-faixa_40-49,143.29,93.765,43.2125,1.146080
2,casos-raca-cor_branca,59.44,26.830,31.1550,1.046702
3,casos-raca-cor_amarela,0.52,0.000,0.5000,1.040000
4,casos-escolaridade_fundamental-completo,15.26,7.990,7.0025,1.038201
5,casos-raca-cor_preta,8.27,3.990,4.3425,0.985607
6,casos-escolaridade_nivel-superior-completo,13.57,7.595,6.3275,0.944291
7,casos-faixa_60-69,226.06,161.775,69.2300,0.928571
8,casos-faixa_50-59,183.03,133.600,57.9800,0.852535
9,casos-escolaridade_fundamental-incompleto,30.76,20.215,14.4475,0.729884



=== Cluster 5 — maiores |d_cj| ===


,VARIAVEL,MEDIANA_CLUSTER,MEDIANA_GERAL,IQR_GERAL,D_CJ
0,casos-escolaridade_nivel-superior-incompleto,15.230,0.575,1.1125,13.173034
1,mamografia_40-49,21515.575,3865.745,3394.8000,5.199078
2,mamografia_20-29,206.910,40.475,51.2575,3.247037
3,mamografia_10-19,27.340,4.455,7.9425,2.881335
4,casos-escolaridade_nivel-medio,28.825,10.895,8.3875,2.137705
5,mamografia_30-39,1227.310,466.970,413.0150,1.840950
6,casos-escolaridade_fundamental-completo,0.000,7.990,7.0025,-1.141021
7,mamografia_70-ou-mais,3277.600,1620.410,1594.4175,1.039370
8,mamografia_50-59,20634.375,12338.030,8389.2475,0.988926
9,casos-raca-cor_amarela,0.485,0.000,0.5000,0.970000


In [34]:
figura_caracterizacao_mediana = grafico_heatmap_caracterizacao(caracterizacao_por_mediana)
figura_caracterizacao_mediana.show()

## Cluster por ano

Se um cluster concentra poucos anos, ele pode estar separando períodos e não regionais.

In [35]:
contagem_cluster_por_ano = tabela_cluster_por_ano(resultado)
print("Contagem de pares Regional-Ano por (CLUSTER, ANO):")
contagem_cluster_por_ano

Contagem de pares Regional-Ano por (CLUSTER, ANO):


ANO,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
CLUSTER,,,,,,,,,,
1,3,2,6,10,8,7,1,2,4,2
2,7,8,6,5,4,7,4,8,6,6
3,13,13,11,7,6,9,15,9,7,7
4,3,2,4,4,6,3,4,6,7,10
5,0,0,0,0,0,1,0,1,1,1


In [36]:
proporcao_cluster_por_ano = contagem_cluster_por_ano.div(contagem_cluster_por_ano.sum(axis=1), axis=0) * 100
print("Proporção por ano dentro de cada cluster (cada linha soma 100%):")
proporcao_cluster_por_ano.round(1)

Proporção por ano dentro de cada cluster (cada linha soma 100%):


ANO,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
CLUSTER,,,,,,,,,,
1,6.7,4.4,13.3,22.2,17.8,15.6,2.2,4.4,8.9,4.4
2,11.5,13.1,9.8,8.2,6.6,11.5,6.6,13.1,9.8,9.8
3,13.4,13.4,11.3,7.2,6.2,9.3,15.5,9.3,7.2,7.2
4,6.1,4.1,8.2,8.2,12.2,6.1,8.2,12.2,14.3,20.4
5,0.0,0.0,0.0,0.0,0.0,25.0,0.0,25.0,25.0,25.0


In [37]:
clusters_por_regional = tabela_clusters_por_regional(resultado)
print("Clusters em que cada Regional aparece, da mais heterogênea pra mais homogênea:")
clusters_por_regional

Clusters em que cada Regional aparece, da mais heterogênea pra mais homogênea:


,REGIONAL DE SAÚDE,CLUSTERS,N_CLUSTERS_DISTINTOS
0,itabira,"[1, 2, 3, 4]",4
1,varginha,"[1, 2, 3, 4]",4
2,manhumirim,"[1, 2, 3]",3
3,barbacena,"[1, 3, 4]",3
4,pirapora,"[1, 2, 3]",3
5,sao joao del rei,"[1, 3, 4]",3
6,teofilo otoni,"[1, 2, 3]",3
7,sete lagoas,"[1, 3, 4]",3
8,passos,"[1, 3, 5]",3
9,patos de minas,"[2, 3, 4]",3


In [38]:
figura_cluster_por_ano = grafico_cluster_por_ano(contagem_cluster_por_ano)
figura_cluster_por_ano.show()

## Cobertura de escolaridade e raça/cor por cluster

Variáveis de escolaridade e raça/cor dependem do preenchimento do registro; zeros concentrados em um cluster sugerem efeito de qualidade do dado e não perfil epidemiológico.

In [39]:
cobertura_por_cluster = resumir_cobertura_por_cluster(matriz, rotulos)
print("Média, mediana e proporção de zeros por (cluster, variável) -- valores originais, antes do RobustScaler:")
cobertura_por_cluster

Média, mediana e proporção de zeros por (cluster, variável) -- valores originais, antes do RobustScaler:


,CLUSTER,VARIAVEL,MEDIA,MEDIANA,PROPORCAO_ZERO
0,1,casos-escolaridade_fundamental-completo,8.008000,7.200,0.000000
1,1,casos-escolaridade_fundamental-incompleto,21.801333,20.270,0.000000
2,1,casos-escolaridade_nenhuma,2.929333,2.550,0.066667
3,1,casos-escolaridade_nivel-medio,12.886667,11.680,0.000000
4,1,casos-escolaridade_nivel-superior-completo,8.007556,7.900,0.000000
5,1,casos-escolaridade_nivel-superior-incompleto,0.957333,0.630,0.177778
6,1,casos-raca-cor_amarela,0.244889,0.000,0.533333
7,1,casos-raca-cor_branca,31.296222,31.670,0.000000
8,1,casos-raca-cor_indigena,0.033333,0.000,0.911111
9,1,casos-raca-cor_parda,22.123333,21.840,0.000000


In [40]:
print("Destaque -- Clusters 4 e 5:")
cobertura_por_cluster[cobertura_por_cluster["CLUSTER"].isin([4, 5])].reset_index(drop=True)

Destaque -- Clusters 4 e 5:


,CLUSTER,VARIAVEL,MEDIA,MEDIANA,PROPORCAO_ZERO
0,4,casos-escolaridade_fundamental-completo,17.889388,15.260,0.000000
1,4,casos-escolaridade_fundamental-incompleto,32.147551,30.760,0.000000
2,4,casos-escolaridade_nenhuma,3.412857,2.880,0.020408
3,4,casos-escolaridade_nivel-medio,21.945510,21.310,0.000000
4,4,casos-escolaridade_nivel-superior-completo,14.100816,13.570,0.000000
5,4,casos-escolaridade_nivel-superior-incompleto,1.572449,1.380,0.102041
6,4,casos-raca-cor_amarela,0.753673,0.520,0.306122
7,4,casos-raca-cor_branca,58.066531,59.440,0.000000
8,4,casos-raca-cor_indigena,0.088367,0.000,0.857143
9,4,casos-raca-cor_parda,27.885918,21.080,0.000000
